# 2. Generating Ground Truth Data

In [1]:
%load_ext autoreload
%autoreload 2
import dotenv

dotenv.load_dotenv(override=True)

True

In [2]:
from src import FaqHttpLoader

loader = FaqHttpLoader()
documents = loader.load()

In [3]:
print(documents[0]['id'])
print(documents[0]['question'])

0e38656cfb
How do I submit homework?


Generating questions with structured output

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.
""".strip()

In [5]:
from openai import OpenAI

ollama_client = OpenAI(
    api_key='ollama',
    base_url='http://localhost:11434/v1',
)

def llm_structured(
    instructions,
    user_prompt,
    output_type,
    model='granite4.1:8b'
    ):
    messages = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=output_type,
        max_tokens=1024,
    )

    return response.choices[0].message.parsed

In [6]:
import json

result = llm_structured(
    data_gen_instructions,
    json.dumps(documents[0]),
    Questions
)

print(result.questions)

['What is the process for completing and submitting homework assignments in the Machine Learning ZoomCamp course?', 'Where should I host my completed homework code before submission?', 'How can I ensure that my submitted answers are visible after the deadline has passed?', 'In which directory of the GitHub repository are the homework materials located for the 2025 cohort?', 'Through which platform do I need to submit my homework?']


Parallel processing

In [7]:
import json
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

CACHE_DIR = Path('../../data/ground_truth')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results


def process(doc):
    cache_file = CACHE_DIR / f"{doc['id']}.json"

    if cache_file.exists():
        return json.loads(cache_file.read_text())

    out = llm_structured(
        data_gen_instructions,
        json.dumps(doc),
        Questions
    )

    results = [
        {'question': q, 'course': doc['course'], 'document': doc['id']}
        for q in out.questions
    ]

    cache_file.write_text(json.dumps(results, ensure_ascii=False, indent=2))
    return results

Generate questions for all documents:

In [9]:
with ThreadPoolExecutor(max_workers=6) as pool:
    ground_truth = map_progress(pool, documents, process)

  0%|          | 0/1208 [00:00<?, ?it/s]

Flatten the nested lists into a single dataset:

In [10]:
import pandas as pd

ground_truth_flat = [item for sublist in ground_truth for item in sublist]
df_ground_truth = pd.DataFrame(ground_truth_flat)

print(len(df_ground_truth))

6020


In [11]:
# Save it for later use:
df_ground_truth.to_csv(Path('../../data')/'ground-truth-data.csv', index=False)


# 3. Search Evaluation

Let's set up our search using RAGBase from module 01:

In [ ]:
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OllamaClient

# Load documents
loader = FaqHttpLoader()
documents = loader.load()

# Index
index = MinsearchIndex(documents)

# LLM-client (Ollama, local)
llm_client = OllamaClient()

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    llm_model='granite4.1:8b',
    instructions=instructions,
)